# Отчаянные домохозяйки в СНТ "Вентилятор-2"

В 5 сезоне одного популярного сериала появилось HOA - HomeOwners Association (некая организация жильцов, которая может решать совместные проблемы, такие как вызов мусора, собирать деньги с жильцов на общие нужды и штрафовать за нарушения (violations) устава). В России такое называется Товарищество Собственников Недвижимости (Жилья), и наиболее популярный вид - Садоводческое Некоммерческое Товарищество (СНТ).

**Наша цель** - разработка кодовой основы под модуль, который реализовывал бы основной функционал HOA/СНТ :)

## Отчаянные домохозяйки - 1

Реализуй следующие классы: `address` (отвечает за непосредственно адреса с улицей, номером дома, корпусом (если есть)), `person` (отвечает за конкретного жителя - его фамилия, имя, отчество; они же в формате Иванов И.И.); `household` - класс, который отвечает за соотношение, какие жильцы где живут, и информацию о конкретных домах (сколько стоит дом, сколько было нарушений у жильцов этого дома, пр.)

Используй dataclass (возможно, с параметрами `frozen=True` или `slots=True`) и property там, где нужно.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True, slots=True)
class Address:
    street: str
    number: int
    building: int = None


@dataclass(slots=True)
class Person:
    surname: str
    name: str
    patronymic: str

    @property
    def full_name(self):
        return f"{self.surname} {self.name[0]}.{self.patronymic[0]}."


@dataclass(slots=True)
class Household:
    address: Address
    tenants: list
    price: float
    violations: int = 0


In [ ]:
first_address = Address("People`s street", 50, 2)
person_one = Person("First", "Man", "Womanowich")
person_two = Person("Second", "Woman", "Manovna")
first_household = Household(first_address, [person_one, person_two], 300, 1)

Какие из классов являются dataclass с какими параметрами? Почему?

<b><font color="white">Я решил сделать dataclass с параметром slots каждый из классов так как они принимают фикисрованное количество атрибутов. Так же класс Adress является frozen так как адрес дома вроде как неизменяемая штука.</font></b>

## Отчаянные домохозяйки - 2

Реализуй класс HOA:
- унаследуй его от `Sequence` из `collections.abc`
- этот класс будет хранить информацию о всех домах (`household`), но итерироваться только по тем, в которых есть жильцы.
- реализуй все методы, необходимые для того, чтобы код заработал.

In [ ]:
from collections.abc import Sequence


class HOA(Sequence):
    def __init__(self, households):
        self.households = households

    def __iter__(self):
        for household in self.households:
            if household.tenants:
                yield household

    def __getitem__(self, index):
        households_with_tenants = [h for h in self.households if h.tenants]
        return households_with_tenants[index]

    def __len__(self):
        return len([h for h in self.households if h.tenants])



In [ ]:
second_household = Household(Address("No peoples`street", 13), [], 5000)
hoa = HOA([first_household, second_household])

for household in hoa:
    print(f"Street: {household.address.street}, Tenants: {len(household.tenants)}")

print(f"All households with tenants: {len(hoa)}")

Street: People`s street, Tenants: 2
All households with tenants: 1


# Борьба с кэшированием

## cache - 1

Допиши строчки так, чтобы декоратор `@cache`, сохраняющий предыдущий вызов функции и возвращающий его, если пришёл тот же аргумент

In [ ]:
def cache(f):
    last_arg = None
    last_res = None
    def wrapper(x):
        nonlocal last_arg
        nonlocal last_res
        if x == last_arg:
            return last_res
        last_arg = x
        last_res = f(x)
        return last_res
    return wrapper

@cache
def f(x):
    for i in range(100000):
        a = i ** 2
    return x ** 2

print(f(1))
print(f(1))

1
1


## cache - 2

Переделай декоратор `@cache` из п.1 так, чтобы он мог работать для функций с произвольным числом аргументов

In [ ]:
def cache(f):
    last_arg = None
    last_res = None
    def wrapper(*args):
        nonlocal last_arg
        nonlocal last_res
        if args == last_arg:
            return last_res
        last_arg = args
        last_res = f(*args)
        return last_res
    return wrapper

@cache
def f(x, y):
    for i in range(100000):
        a = i ** 2
    return (x ** 2) * y

print(f(1, 2))
print(f(1, 2))

2
2


##LRU_cache - 1

Реализуй декоратор `@lru_cache`: он будет хранить результаты последних 5 уникальных вызовов функции.

- Если мы вызвали то, что уже лежит в наборе последних 5 уникальных вызовов, то поднимаем "вверх" этот вызов до последнего и выводим результат
- Если мы вызвали то, что не лежит в наборе:

- если длина набора меньше 5, то просто добавляем последний вызов, считаем значение и выводим
- если длина набора 5, то удаляем самый старый вызов, добавляем наш вызов с учётом посчитанного значения и выводим



In [ ]:
def lru_cache(func):
    last_five_args = ()
    last_five_res = ()
    def wrapper(*args):
        nonlocal last_five_args
        nonlocal last_five_res
        if args in last_five_args:
            index = last_five_args.index(args)
            result = last_five_res[index]
            last_five_args = last_five_args[:index] + last_five_args[index+1:] + (args,)
            last_five_res = last_five_res[:index] + last_five_res[index+1:] + (result,)
            return result
        else:
            result = func(*args)
            if len(last_five_args) < 5:
                last_five_args = last_five_args + (args,)
                last_five_res = last_five_res + (result,)
            else:
                last_five_args = last_five_args[1:] + (args,)
                last_five_res = last_five_res[1:] + (result,)
            return result
    return wrapper

@lru_cache
def f(x, y):
    for i in range(100000):
        a = i ** 2
    return (x ** 2) * y

print(f(1, 2))
print(f(1, 2))

2
2


## LRU_cache - 2

Возьми код из предыдущей ячейки и реализуй `@lru_cache(n)`, которых хранит результаты последних n уникальных вызовов функции.

In [ ]:
def lru_cache(n):
    def first_wrapper(func):
        last_args = ()
        last_res = ()
        def second_wrapper(*args):
            nonlocal last_args
            nonlocal last_res
            if args in last_args:
                index = last_args.index(args)
                result = last_res[index]
                last_args = last_args[:index] + last_args[index+1:] + (args,)
                last_res = last_res[:index] + last_res[index+1:] + (result,)
                return result
            else:
                result = func(*args)
                if len(last_args) < n:
                    last_args = last_args + (args,)
                    last_res = last_res + (result,)
                else:
                    last_args = last_args[1:] + (args,)
                    last_res = last_res[1:] + (result,)
                return result
        return second_wrapper
    return first_wrapper

@lru_cache(10)
def f(x, y):
    for i in range(100000):
        a = i ** 2
    return (x ** 2) * y

print(f(1, 2))
print(f(1, 2))

2
2


Можем ли мы корректно удалять объекты, которые возвращает lru_cache? Почему?


<b><font color="white">Возвращаемые объекты хранятся в замыкании поэтому не можем</font></b>

## LRU_cache - 3

Реализуй `LRU_cache(n)` через `weakref`, чтобы можно было удалять элементы.

In [ ]:
import weakref as wr


class AbleToWeak:
  def __init__(self, elem):
    self.elem = elem


def lru_cache(n):
    def first_wrapper(func):
        last_args = ()
        last_res_refs = ()
        def second_wrapper(*args):
            nonlocal last_args
            nonlocal last_res_refs
            if args in last_args:
                index = last_args.index(args)
                result = last_res_refs[index]()
                if result is not None:
                    last_args = last_args[:index] + last_args[index+1:] + (args,)
                    last_res_refs = last_res_refs[:index] + last_res_refs[index+1:] + (last_res_refs[index],)
                    return result.elem
                else:
                    last_args = last_args[:index] + last_args[index+1:]
                    last_res_refs = last_res_refs[:index] + last_res_refs[index+1:]
            result = AbleToWeak(func(*args))
            res_ref = wr.ref(result)
            if len(last_args) < n:
                last_args = last_args + (args,)
                last_res_refs = last_res_refs + (res_ref,)
            else:
                last_args = last_args[1:] + (args,)
                last_res_refs = last_res_refs[1:] + (res_ref,)
            return result.elem
        return second_wrapper
    return first_wrapper

@lru_cache(10)
def f(x, y):
    for i in range(100000):
        a = i ** 2
    return (x ** 2) * y

print(f(1, 2))
print(f(1, 2))

2
2


# Скитания Palomar-1 часть 2

## Генератор и/или декоратор Palomar - 1

Возьми датасет про Palomar-1. Давай сымитируем передачу данных между сервисами - напиши класс, который в `__init__` получает название файла (путь до файла), в `read_from_file` записывает все данные из файла в некоторый атрибут, а в `get_lines(n)` возвращает следующие n линий из файла в формате np.array (такой вот итератор без `__iter__` и `__next__`).

Если линии кончились - пусть метод выбрасывает ошибку (StopIteration).

Протестируй работу класса.

In [ ]:
import pandas as pd
import numpy as np


class DataReader:
    def __init__(self, way):
        self.way = way
        self.data = None
        self.ind = 0
        self.border = False

    @property
    def read_from_file(self):
        columns = ['t', 'X', 'Y', 'Z', 'U', 'V', 'W']
        self.data = pd.read_csv("/content/sample_data/Pal1_new.dat", skiprows=[0], names=columns, delimiter=' ')

    def get_lines(self, n):
        if self.border:
          raise StopIteration
        if (self.ind + n) >= len(self.data) and not self.border:
          self.border = True
          return np.array(self.data[self.ind:])
        self.ind += n
        return np.array(self.data[self.ind - n:self.ind])


In [ ]:
mass = DataReader('/content/sample_data/Pal1_new.dat')
mass.read_from_file
print(mass.get_lines(10))
print(mass.get_lines(5))
print(mass.get_lines(10000000))
# print(mass.get_lines(5))
# Если первые 3 строки то последовательно соберется весь файл и без ошибки
# Добавляем четвертую и появляется ошибка

[[  0.         -13.43070444   6.45981983   2.9210541   62.24441343
  208.7595179  -23.26419995]
 [  1.         -13.36563023   6.67262558   2.89666246  65.01279852
  207.40279129 -24.43533157]
 [  2.         -13.29773635   6.88402367   2.87107756  67.75860017
  206.00673688 -25.59774446]
 [  3.         -13.22704605   7.09397413   2.84430839  70.48148575
  204.57184723 -26.75131105]
 [  4.         -13.15358294   7.30243749   2.81636406  73.1811319
  203.09861205 -27.8959071 ]
 [  5.         -13.07737095   7.50937479   2.7872538   75.85722417
  201.58751807 -29.03141163]
 [  6.         -12.99843433   7.71474756   2.75698697  78.50945684
  200.03904887 -30.15770674]
 [  7.         -12.91679762   7.91851782   2.72557305  81.13753255
  198.45368477 -31.27467752]
 [  8.         -12.83248568   8.12064808   2.69302163  83.74116209
  196.83190269 -32.38221191]
 [  9.         -12.74552366   8.32110134   2.65934241  86.32006409
  195.17417606 -33.48020056]]
[[ 10.         -12.65593699   8.51984108

## Генератор и/или декоратор Palomar - 2

Напиши функцию palomar_mean, которая бы принимала `np.array` из нескольких строк из Palomar'a и считает среднее по каждому из них, возвращая другой `np.array`.

In [ ]:
def palomar_mean(incoming_array):
    return np.mean(incoming_array, axis=0)

In [ ]:
array_one = DataReader('/content/sample_data/Pal1_new.dat')
array_one.read_from_file
data_one = array_one.get_lines(100)
array_two = DataReader('/content/sample_data/Pal1_new.dat')
array_two.read_from_file
data_two = array_two.get_lines(10)
print(palomar_mean(data_one))
print(palomar_mean(data_two))

[ 49.5         -6.7482192   13.50428005   0.36047507 153.5640145
  96.38003867 -53.70073466]
[  4.5        -13.10453123   7.39972703   2.79716444  74.42438695
 202.19258358 -28.42707025]


## Генератор и/или декоратор Palomar - 3

Напиши декоратор `generate_rows(n)`, который:

- принимает на вход параметр n (по сколько строк данных он должен брать) - для метода `get_lines` у класса `DataReader`

- принимает функцию f, а на выходе выдаёт генератор, который возвращает обработанные функцией f строки `np.array`.

In [ ]:
def generate_rows(n):
    def wrapper(f):
        def generator(data):
          data.ind = 0
          data.border = False
          while True:
              try:
                  next_data = data.get_lines(n)
                  yield f(next_data)
              except StopIteration:
                  break
        return generator
    return wrapper


In [ ]:
k = 100

@generate_rows(k)
def palomar_mean(incoming_array):
    return np.mean(incoming_array, axis=0)

array_one = DataReader('/content/sample_data/Pal1_new.dat')
array_one.read_from_file
elems = palomar_mean(array_one)
ind = 1
for elem in elems:
  print(f"Разбиаение по {k} строкам, группа № {ind}")
  print(elem)
  ind += 1

Разбиаение по 100 строкам, группа № 1
[ 49.5         -6.7482192   13.50428005   0.36047507 153.5640145
  96.38003867 -53.70073466]
Разбиаение по 100 строкам, группа № 2
[ 149.5           9.91963656    9.78599887   -2.80498748  115.06936514
 -158.21268024   14.22736652]
Разбиаение по 100 строкам, группа № 3
[ 249.5           7.77921063   -9.30889395    1.87400669 -157.22420567
 -142.53233154   39.02813791]
Разбиаение по 100 строкам, группа № 4
[ 3.49500000e+02 -1.09696554e+01 -1.01960288e+01  2.70126938e-01
 -1.38498553e+02  1.19016496e+02 -5.40387817e+01]
Разбиаение по 100 строкам, группа № 5
[449.5        -12.40134615   6.76819561  -2.84166373 115.91822625
 154.94577357  14.72498835]
Разбиаение по 100 строкам, группа № 6
[ 549.5           6.36268852   10.25838458    1.83962126  182.77019212
 -108.98073948   38.92200267]
Разбиаение по 100 строкам, группа № 7
[ 6.49500000e+02  1.27607627e+01 -7.56756425e+00  1.87242491e-01
 -7.33486189e+01 -1.68883309e+02 -5.43781064e+01]
Разбиаение по 

# Большой брат `observer` за тобой

## Observer - 1


Напиши класс `ObservableMixin`. Он должен:

- Переопределять `__init__`, при этом не нарушать наследование (не забудь про super())
- Создавать словарь с observer'ами в init'е - по сути ссылками на функции, которые надо будет дёргать. Ключ - строка - название события, по которому надо начать оповещать. Значение - список из функций, которые надо будет вызвать в случае события.

У него должны быть методы:
- add_observer - добавление обзервера (функций для оповещения) для выбранных событий

- delete_observer - удаление обзервера (функции для оповещения) среди всех событий

- clear - очищение всех обзерверов

- notify_observers - запускает все функции из списка, который хранится по данному ключу

In [ ]:
class ObservableMixin:
    def __init__(self):
        super().__init__()
        self._observers = {}

    def add_observer(self, event, observer):
        if event not in self._observers:
            self._observers[event] = []
        self._observers[event].append(observer)

    def delete_observer(self, observer):
        for observers in self._observers.values():
            if observer in observers:
                observers.remove(observer)

    def clear(self):
        self._observers.clear()

    def notify_observers(self, event, *args, **kwargs):
        find = False
        if event in self._observers:
            for observer in self._observers[event]:
                observer(*args, **kwargs)
                find = True
            if not find:
                print("There is no such observer")
        else:
            print("There is no such event")

In [ ]:
class Button(ObservableMixin):
    def __init__(self, label: str):
        super().__init__()
        self.label = label
        self.clicked = False

    def click(self) -> None:
        self.clicked = True
        self.notify_observers("click", button=self)

In [ ]:
def on_button_click(button):
    print(f"Button '{button.label}' was clicked!")

button1 = Button("Subsribe")
button1.add_observer("click", on_button_click)
button2 = Button("Repost")
button2.add_observer("click", on_button_click)
button3 = Button("Like")
button3.add_observer("click", on_button_click)
button3.click()
button2.click()
button1.click()
button2.delete_observer(on_button_click)
button2.click()
button1.click()
button1.clear()
button1.click()

Button 'Like' was clicked!
Button 'Repost' was clicked!
Button 'Subsribe' was clicked!
There is no such observer
Button 'Subsribe' was clicked!
There is no such event


## Observer - 2

Теперь реализуй декоратор observable, который на вход принимает класс, который нужно сделать observable - реализует Observable (по своей сути ObservableMixin из задания выше), наследуемый от него, и переопределяет методы так, чтобы они после своего выполнения запускали `notify_observers`.

In [ ]:
def observable(cls):
    class Observable(cls):
        def __init__(self, *args, **kwargs):
            self._observers = {}
            cls.__init__(self, *args, **kwargs)

        def add_observer(self, event, observer):
            if event not in self._observers:
                self._observers[event] = []
            self._observers[event].append(observer)

        def delete_observer(self, observer):
            for observers in self._observers.values():
                if observer in observers:
                    observers.remove(observer)

        def clear(self):
            self._observers.clear()

        def notify_observers(self, event, *args, **kwargs):
            find = False
            if event in self._observers:
                for observer in self._observers[event]:
                    observer(*args, **kwargs)
                    find = True
                if not find:
                    print("There is no such observer")
            else:
                print("There is no such event")


    for name in dir(cls):
        if name[0] != '_' and callable(getattr(cls, name)):
            orig = getattr(cls, name)
            def wrapper(self, *args, **kwargs):
                result = orig(self, *args, **kwargs)
                self.notify_observers(name, name, *args, **kwargs)
                return result
            setattr(Observable, name, wrapper)
    return Observable

In [ ]:
@observable
class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount
        return self.balance

def logger(method_name, *args, **kwargs):
    print(f"Method '{method_name}' called with args: {args}, kwargs: {kwargs}")

my_acc = BankAccount(1000)
my_acc.add_observer("deposit", logger)
my_acc.deposit(2000)

# тут какой-то код для проверки, но ты можешь придумать свой код для проверки :)

Method 'deposit' called with args: (2000,), kwargs: {}


3000

Теперь:

1. Сравни эти две реализации. Какая лучше?
2. Предложи свои идеи по улучшению/доработке. Что можно было бы реализовать по-другому?

<b><font color="white">Ну первый вариант проще в реализации и наверное понятнее. Вторая реализация лучше тем что по сути все автоматизировано. Не надо лишних строк кода, поэтому если все сделать аккуратно, наверное вторая лучше. Во-втором варианте может, кажется, случится повторная обертка, а еще имя функции теряется, да и в общем-то всё описание исходной. Поэтому можно это поправить.</font></b>

# Удивительный `descriptor` страны Оз

Дескрипторы в Python - классы, в которых реализованы методы `__get__`, `__set__` или `__del__`. Эти методы вызываются, когда мы обращаемся к объекту данного класса как к атрибуту у другого класса.

In [ ]:
# пример, тут делать ничего не надо

class Ten:
    def __get__(self, *args, **kwargs):
        return 10

class Numbers:
    a = 5
    ten = Ten()


print(Numbers.a, Numbers.ten)

5 10


Оказывается, что многие фишки реализованы через дескрипторы в питоне. Ваша задача - написать свои аналоги (быть может, упрощенные версии того, как этот код работает на самом деле)

## @my_property

Реализуй @my_property - аналог @property - через дескрипторы

In [ ]:
class my_property:
  def __init__(self, fget):
    self.fget = fget
    self.fset = None

  def __get__(self, instance, owner):
    if instance is None:
      return self
    return self.fget(instance)

  def __set__(self, instance, value):
    if self.fset is None:
      raise AttributeError("No possibility of change")
    return self.fset(instance, value)

  def setter(self, fset):
    self.fset = fset
    return self

In [ ]:
class Fruit:
    def __init__(self, color):
        self.color = color

    @my_property
    def get_color(self):
        return self.color

    @get_color.setter
    def change_color(self, new_color):
        self.color = new_color
        return self.color


apple = Fruit('red')
print(apple.get_color)
apple.change_color = 'green'
print(apple.get_color)

red
green


## @my_classmethod

Реализуй @my_classmethod - аналог @classmethod - через дескрипторы

In [ ]:
class my_classmethod:
    def __init__(self, fget):
        self.fget = fget

    def __get__(self, instance, owner):
        def wrapper(*args, **kwargs):
            return self.fget(owner, *args, **kwargs)
        return wrapper

In [ ]:
class Date:
    def __init__(self, years, months, days):
        self.years = years
        self.months = months
        self.days = days

    @classmethod
    def today(cls, value):
        y, m, d = list(map(int, value.split()))
        return cls(y, m, d)

tdy = "2025 09 20"
now = Date.today(tdy)
print(f"Today is {now.days} {now.months} {now.years}")

Today is 20 9 2025


## @my_staticmethod

Реализуй @my_staticmethod - аналог @staticmethod - через дескрипторы

In [ ]:
class my_staticmethod:
    def __init__(self, fget):
        self.fget = fget

    def __get__(self, instance, owner):
        def wrapper(*args, **kwargs):
            return self.fget(*args, **kwargs)
        return wrapper

In [ ]:
class NameOfClass:

  @my_staticmethod
  def greatings():
    return f"Hello, my dear user. My name is {NameOfClass.__name__}"

copy = NameOfClass()
print(copy.greatings())
print(NameOfClass.greatings())

Hello, my dear user. My name is NameOfClass
Hello, my dear user. My name is NameOfClass


# Джонни! `Singleton`'ы на деревьях!

## Singleton - Dec1

Допиши код внизу, чтобы он стал рабочим

In [ ]:
def singleton(cls):
    _instances = {}

    def return_obj(*args, **kwargs):
        if cls not in _instances:
            _instances[cls] = cls(*args, **kwargs)
        return _instances[cls]
    return return_obj

@singleton
class House:
    def __init__(self, name):
        self.name = name

    def whose(self):
        print(f"It's {self.name}'s house!")

my_name = "Khodyrev Mikhail"
h1 = House(my_name)
h1.whose()
h2 = House(f"Not {my_name}")
h2.whose()
print(h1 is h2)

It's Khodyrev Mikhail's house!
It's Khodyrev Mikhail's house!
True


Данная реализация позволяет нам вызывать повторно `__init__`?

<b><font color="white">Нет так как при следующих попытках мы просто идем в instances где уже есть ключ наш класс. Поэтому просто возвращается копия которая там хранится. Собственно поэтому и имя не изменилось.</font></b>

## Singleton - Dec2

Реализуй декоратор так, чтобы он позволял не создавать новый объект, но вызывал повторно `__init__`

In [ ]:
def singleton(cls):
    _instances = {}

    def return_obj(*args, **kwargs):
        if cls not in _instances:
            _instances[cls] = cls(*args, **kwargs)
            return _instances[cls]
        _instances[cls].__init__(*args, **kwargs)
        return _instances[cls]
    return return_obj

@singleton
class House:
    def __init__(self, name):
        self.name = name

    def whose(self):
        print(f"It's {self.name}'s house!")

my_name = 'Jedi'
h1 = House(f"Not {my_name}")
h1.whose()
h2 = House(f"{my_name}")
h2.whose()
print(h1 is h2)

It's Not Jedi's house!
It's Jedi's house!
True


## Singleton - Dec3

Обязательно ли нам нужен мутабельный объект (`dict`, `list`) для реализации декоратора `singleton`?

Реализуй декоратор через просто хранение переменной `_instance`.

In [ ]:
def singleton(cls):
    _instances = None

    def return_obj(*args, **kwargs):
        nonlocal _instances
        if _instances is None:
            _instances = cls(*args, **kwargs)
            return _instances
        _instances.__init__(*args, **kwargs)
        return _instances
    return return_obj

@singleton
class House:
    def __init__(self, name):
        self.name = name

    def whose(self):
        print(f"It's {self.name}'s house!")


my_name = 'Yoda'
h1 = House(f"Not {my_name}")
h1.whose()
h2 = House(f"{my_name}")
h2.whose()
print(h1 is h2)
del h1
h2.whose()

It's Not Yoda's house!
It's Yoda's house!
True
It's Yoda's house!


Корректен ли такой код? Если да, то почему? Если нет - приведи контрпример, где некорректен


<b><font color="white">Ну вроде как определению синглтона удовлетворяет. Экземпляр класса действительно один. Однако отвечая на вопрос из следующего блока удалять его корректно не получается, потому что удалем ссылки по сути, а их количество не регламентировано.</font></b>

## Singleton - Dec4

Получается ли удалять объекты корректно с таким декоратором? Почему?

Реализуй декоратор через `weakref`, чтобы поддерживать удаление

In [ ]:
import sys


def singleton(cls):
    _instances = None

    def return_obj(*args, **kwargs):
        nonlocal _instances
        if _instances is None:
            _instances = cls(*args, **kwargs)
            return wr.ref(_instances)()
        _instances.__init__(*args, **kwargs)
        return wr.ref(_instances)()
    return wr.ref(return_obj)()


@singleton
class House:
    def __init__(self, name):
        self.name = name

    def whose(self):
        print(f"It's {self.name}'s house!")


my_name = 'Yoda'
h1 = House(f"Not {my_name}")
h1.whose()
h2 = House(f"{my_name}")
h2.whose()
print(h1 is h2)
del h1
h2.whose()
sys.getrefcount(h2)

It's Not Yoda's house!
It's Yoda's house!
True
It's Yoda's house!


3

## Singleton - Meta1

Допиши код внизу, чтобы он был рабочим

In [ ]:
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class House(metaclass=SingletonMeta):
    def __init__(self):
        print("It's my House!")

h1 = House()
h2 = House()

It's my House!


## Singleton - Meta2

Можно ли реализовать код выше не через mutable объект (dict), а через изменение одной переменной?

1. Попробуй сделать это через `nonlocal _instance`. Получается?

In [ ]:
class SingletonMeta(type):
    _instances = None

    def __call__(cls, *args, **kwargs):
        nonlocal _instances
        if cls._instances is None:
            cls._instances = super().__call__(*args, **kwargs)
        return cls._instances

class House(metaclass=SingletonMeta):
    def __init__(self):
        print("It's my House!")

class People(metaclass=SingletonMeta):
    def __init__(self):
        print("It's me!")

h1 = House()
h2 = House()
print(h1 is h2)
p1 = People()
p2 = People()
print(h1 is p1)
# при таком же подходе просто с изменением одной переменной не получается (так как по LEGB не находится nonlocal), но если сделать атрибутом класса то получится

SyntaxError: no binding for nonlocal '_instances' found (ipython-input-39281279.py, line 5)

In [ ]:
class SingletonMeta(type):
    def __init__(cls, name, bases, namespace, *args, **kwargs):
        super().__init__(name, bases, namespace, *args, **kwargs)
        cls._instances = None

    def __call__(cls, *args, **kwargs):
        if cls._instances is None:
            cls._instances = super().__call__(*args, **kwargs)
        return cls._instances

class House(metaclass=SingletonMeta):
    def __init__(self):
        print("It's my House!")

class People(metaclass=SingletonMeta):
    def __init__(self):
        print("It's me!")

h1 = House()
h2 = House()
print(h1 is h2)
p1 = People()
p2 = People()
print(h1 is p1)
# вот так например

It's my House!
True
It's me!
False


Объясните, почему получилось или не получилось

2. Попробуй сделать это через `SingletonMeta._instance`. Корректен ли код?

Если нет - приведи ситуацию, когда код ведёт себя некорректно

In [ ]:
class SingletonMeta(type):
    _instances = None

    def __call__(cls, *args, **kwargs):
        if SingletonMeta._instances is None:
            SingletonMeta._instances = super().__call__(*args, **kwargs)
        return SingletonMeta._instances

class House(metaclass=SingletonMeta):
    def __init__(self):
        print("It's my House!")

class People(metaclass=SingletonMeta):
    def __init__(self):
        print("It's me!")

h1 = House()
h2 = House()
print(h1 is h2)
p1 = People()
p2 = People()
print(h1 is p1)

It's my House!
True
True


Опиши, что пошло не так. Можно ли было так сделать, когда мы писали через декоратор? Почему?

<b><font color="white">Из-за того что instances общий то и синглтон один на всех(наследуемся от одного метакласса), не круто. Когда писали декоратор там обертка создавала для каждого класса свой instances, поэтому такой проблемы возникнуть не должно было.</font></b>

## Singleton - Meta3

Какие есть минусы у текущей реализации `Singleton` через Метаклассы? Позволяет ли он удалять объекты? Почему?



<b><font color="white">Ну вот из минусов получается было непонятно куда и как засунуть instances, но даже в реализации где все норм удалять не получится, так как есть всегда атрибут метакласса, который не удаляется.</font></b>

Реализуй удаление через метод `delete_object` (т.е. мы глобально хотим вызывать `h.delete_object()`). Где он (метод) должен находиться, чтобы он работал?

In [ ]:
class SingletonMeta(type):
    _instances = {}
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]


class House(metaclass=SingletonMeta):
    def __init__(self):
        print("It's my House!")

    @classmethod
    def delete_object(cls):
        if cls in cls._instances:
            del cls._instances[cls]
            print(f"Экземпляр класса {cls.__name__} удален")


class People(metaclass=SingletonMeta):
    def __init__(self):
        print("It's me!")

    @classmethod
    def delete_object(cls):
        if cls in cls._instances:
            del cls._instances[cls]
            print(f"Экземпляр класса {cls.__name__} удален")


h1 = House()
h2 = House()
print(h1 is h2)
p1 = People()
p2 = People()
print(h1 is p1)
People.delete_object()

It's my House!
True
It's me!
False
Экземпляр класса People удален


Объясни, как работает твоё решение

<b><font color="">Вызывая метод проверяем есть ли экземпляр в словаре. Если есть удаляем.</font></b>

## Singleton - Meta4

Реализуй хранение в метаклассе через `weakref`.

In [ ]:
import weakref

class SingletonMeta(type):
    _instances = weakref.WeakValueDictionary()
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            instance = super().__call__(*args, **kwargs)
            cls._instances[cls] = instance
        return cls._instances[cls]


class House(metaclass=SingletonMeta):
    def __init__(self):
        print("It's my House!")


class People(metaclass=SingletonMeta):
    def __init__(self):
        print("It's me!")


h1 = House()
h2 = House()
print(h1 is h2)
p1 = People()
p2 = People()
print(h1 is p1)

It's my House!
True
It's me!
False


Какое решение лучше для удаления объектов - в Meta3 (через некий метод) или в Meta4 (через `weakref`)?

Сравните их. В какой ситуации какой может быть лучше?


<b><font color="white">Meta3 даёт больше контроля над памятью, так как ни один объект не будет удален автоматически.Все делается в ручную. Если такой контроль необходим, то он лучше. В остальном лучше наверное автоматизированный Met4.</font></b>

## Singleton'ы. Сравнение

Сравни реализации Singleton'ов через декораторы и через метаклассы. Какие есть плюсы и минусы? Что лучше?

<b><font color="white">Метаклассы если можно так сказать  фундаментальнее, а декораторы динамичнее, можно применить к конкретному классу, изменения почти налету. А вот метаклассы лучше использовать когда выстраивается сложная структура наследования.</font></b>

# `__slots__` этого `__attribute_name__` казино

Как мы знаем, `__slots__` лучше, чем `__dict__` в контексте времени и памяти, но убивает динамичность атрибутов.

Объясни своими словами, какая проблема возникнет, если мы захотим сделать метакласс, который будет нам перестраивать классы таким образом?

<b><font color="white">Проблемы с наследованием наверное возникнут. Если мы наследуемся от класса со slots то непонятно что у дочернего будет, как там добавить новые атрибуты. В обратную сторону нужно проверять что все родительские классы уже используют slots. Логика какая-то неявная появляется, что может приводить к ошибкам.</font></b>

## MakeSlottable - Meta1

Давай попробуем передавать аргументы в метаклассы. Наша первая идея - поступить как с декораторами - сделать функцию, которая станет замыканием для наших методов класса; таким образом, на каждый вызов у нас будет свой метакласс.

In [ ]:
def MakeSlottable(*slots_names):
    class Slottable(type):
        def __new__(cls, name, bases, namespace, *args, **kwargs):
            namespace['__slots__'] = slots_names
            return super().__new__(cls, name, bases, namespace, *args, **kwargs)
    return Slottable

In [ ]:
class A(metaclass = MakeSlottable("a", "b", "c")):
    def __init__(self, a, b, c):
        self.a = a
        self.b = b
        self.c = c

In [ ]:
a = A(1, 2, 3)
print(a.a, a.b, a.c)
print(hasattr(a, '__dict__'))

1 2 3
False


## MakeSlottable - Meta2

Теперь давай переделаем этот метакласс: будем передавать параметры через kwargs при наследовании.

In [ ]:
class MakeSlottable(type):
    def __new__(cls, name, bases, namespace, *args, **kwargs):
        slots_names = kwargs.pop('slots_names', [])
        namespace['__slots__'] = slots_names
        return super().__new__(cls, name, bases, namespace, *args, **kwargs)

In [ ]:
class A(metaclass = MakeSlottable, slots_names = ["a", "b", "c"]):
    def __init__(self, a, b, c):
        self.a = a
        self.b = b
        self.c = c

In [ ]:
a = A(1, 2, 3)
print(a.a, a.b, a.c)
print(hasattr(a, '__dict__'))

1 2 3
False


Сравни эти два способа реализации передачи параметров в метакласс. Какой лучше? Найди **минимум 2** различных минуса у первого (Meta1) способа.

<b><font color="white">В первом способе при каждом вызове функции мы создаем новый метакласс, что избыточно. Это излишнее потребление памяти, усложнение логики наследования, ведь теперь у каждого класса даже с вызванной с одинаковыми аргументами функции, разные метаклассы.</font></b>

## MakeSlottable - Meta3



Давай попробуем реализовать без каких-либо параметров - метакласс, который бы из этой записи (через атрибуты класса):
```
class MyClass:
    x: int
    y: int
    z: int
```

`__slots__` с `x`, `y`, `z` вместо `__dict__`

In [ ]:
class MakeSlottable(type):
    def __new__(cls, name, bases, namespace, *args, **kwargs):
        slots = (i for i in namespace['__annotations__'])
        namespace['__slots__'] = slots
        return super().__new__(cls, name, bases, namespace, *args, **kwargs)

In [ ]:
class A(metaclass = MakeSlottable):
    a: None
    b: None
    c: None
    def __init__(self, a, b, c):
        self.a = a
        self.b = b
        self.c = c

In [ ]:
a = A(1, 2, 3)
print(a.a, a.b, a.c)
print(hasattr(a, '__dict__'))

1 2 3
False


## MakeSlottable - Meta4

Давай попробуем теперь использовать `inspect.sig` из модуля `inspect` для получения атрибутов init'а, чтобы на их основе сделать `slots`.

In [ ]:
import inspect

class MakeSlottAble(type):
    def __new__(cls, name, bases, namespace, *args, **kwargs):
        signature = inspect.signature(namespace['__init__'])
        slots = ()
        for name, param in signature.parameters.items():
            slots += (name,)
        namespace['__slots__'] = slots
        return super().__new__(cls, name, bases, namespace, *args, **kwargs)

In [ ]:
class A(metaclass = MakeSlottAble):
    def __init__(self, a, b, c):
        self.a = a
        self.b = b
        self.c = c

In [ ]:
a = A(1, 2, 3)
print(a.a, a.b, a.c)
print(hasattr(a, '__dict__'))

1 2 3
False


Сравни два способа реализации MakeSlottable без аргументов (Meta3 и Meta4). Какой лучше? Для каких целей?

Сравни их со способами с аргументами. Какой бы ты выбрал, если бы нужно было сделать один вариант для прода?

<b><font color="white">Думаю лучше Meta3 так как заранее прописаны все атрибуты класса. Он надежнее. По этим же причинам для реальных задач наверное выбрал бы Meta2. Там все ручками. В прод пошел бы вообще класс где декоратор dataclass с параметром slots=True.</font></b>

# `N-Singleton` strikes back

Реализуй метакласс `nsingleton`, который получает на вход n - максимальное количество экземпляров класса, которые могут существовать одновременно в ходе выполнения программы.

В случае, если у нас ещё нет n экземпляров класса, то он создаёт новый; если уже есть n экземпляров, то возвращает один из предыдущих, причём он итерируется по ним при последовательных вызовах(сначала вернёт первый экземпляр, потом второй и.т.д.)

In [ ]:
class Nsingleton(type):
    def __new__(mcls, name, bases, namespace, **kwargs):
        n = int(kwargs.pop('n', 1))

        cls = super().__new__(mcls, name, bases, namespace)
        cls._instances = []
        cls._ind = 0
        cls._n = n
        return cls

    def __call__(cls, *args, **kwargs):
        if len(cls._instances) < cls._n:
            instance = super().__call__(*args, **kwargs)
            cls._instances.append(instance)
            return instance
        instance = cls._instances[cls._ind]
        cls._ind = (cls._ind + 1) % cls._n
        return instance